# Tunisian Derja ASR v3 — Whisper-small fine-tuning, leakage-safe evaluation, low-latency inference (Kaggle)

**How to run**
1. Kaggle settings: GPU (T4 / T4×2 / P100), Internet **on**. Optional secret `HF_TOKEN`.
2. **Smoke test (default):** `RUN_MODE = "smoke"` → *Run All*. ~25–40 min. Checks data loading, forward/backward, generation, metrics, checkpointing, CT2 export, streaming. Smoke scores mean nothing.
3. **Full run:** set `RUN_MODE = "full"`, *Restart & Run All* (or *Save Version → Run All* so it survives closing the browser). The step count is computed from a **measured** throughput benchmark so that training + evaluation + export fit in `SESSION_LIMIT_H` with a reserve.
4. **Resume:** keep `RESUME = "auto"`. If `/kaggle/working/whisper-tn-v3/checkpoint-*` exists, training continues from the latest checkpoint with the same saved plan (`run_plan.json`). In a *new* session, first attach the previous version's output as an input dataset and set `RESUME_FROM_DIR` to it.

**Why v1's held-out WER (56.8 %) was much worse than its validation WER (43.9 %)** — evidence from the v1 outputs and the dataset cards:

| # | Finding | Evidence | Fix in v3 |
|---|---|---|---|
| 1 | **Validation and test were different domains.** TunSwitchCS = spontaneous radio/podcast speech with ~27 % French/English words; TunSwitchTO = *read* crowd-sourced sentences, richest vocabulary, very varied mics. Selecting a checkpoint on CS says little about TO. | TunSwitch paper / Zenodo card | Dev and test are drawn from the **same pool of all original test splits**, split **by recording group** (hash), so they share a distribution but not recordings. Results are also reported **per sub-corpus**. |
| 2 | **Checkpoint selection on a tiny, noisy dev set.** 240 segments from a 32-row shuffle buffer = few recordings. Dev WER jumped 66→62→**44**→64→62→54→47 between checkpoints: the "best" (step 1200) is partly a lucky draw, so its dev score is optimistic (winner's curse). | v1 training table | Larger dev set (≤400 utts, ≤4 per recording), cluster-bootstrap 95 % CIs, ~10 evaluations per run. |
| 3 | **Non-i.i.d. training stream.** The augmented `default` config is the 16 augmented subsets concatenated file by file (starts with 6 k single-word *ApprendreLeTunisien* items). One stream + 32-row buffer ⇒ the model sees one sub-corpus for hundreds of steps; dev WER oscillates with whatever it was just trained on. Training loss never decreased steadily (1.24→0.94→0.83→0.99→0.64). | dataset viewer; v1 loss/WER curve | 4 worker streams on **different shards** (shard-order shuffled), interleaved with the original corpus, ≤4 segments per recording, feature-level shuffle buffer. |
| 4 | **Data loading was the bottleneck; 3 of 4 workers were silently stopped.** `IterableDataset.from_generator` has 1 shard → "Stopping 3 dataloader workers". 2400 steps took 6 h 49 m ≈ 10 s/step. | v1 warnings | Custom shard-aware PyTorch `IterableDataset`; workers actually decode in parallel. |
| 5 | **Two GPUs were in use without being accounted for.** The repeated *"gather along dimension 0"* warning and the eval progress bar `120/60` are `nn.DataParallel` signatures ⇒ effective batch was 32, not 16. | v1 warnings | GPU count detected and printed; effective batch computed from it; `GPU_MODE` switch. |
| 6 | **Large parts of the corpus were never used.** The augmented repo contains only VCA (voice-conversion augmented) versions of a subset of corpora; the original repo's *AmenyKH*, *Wav2Vec-tunisian-Darja*, *Tunisian_dataset_STT-TTS15s* etc. are absent. | both dataset cards | Train on a mixture: 60 % augmented + 40 % original `default/train`. |
| 7 | **Leakage risk.** VCA keeps the text and changes the voice. If a test utterance was augmented into the train repo, the model has memorised its exact transcript. | VCA definition | Before training, eval transcripts are put on a **block-list**; any training recording containing an eval sentence (≥3 words) is dropped and counted. An optional text-only audit quantifies overlap up-front. |
| 8 | **Misaligned segments.** `No audio frames were decoded … start_seconds is too high (603 s)`: segment timestamps beyond the end of the audio. The *other* segments of that recording are suspect too. | v1 warnings | Recording duration checked from metadata; a recording with any out-of-range timestamp is rejected entirely; decoded length must match the timestamp span; reasons counted. |
| 9 | **Normalization hid nothing but also missed common variants.** ة/ه, standalone و vs attached و (`و مشيت` / `ومشيت` appear in both refs and hyps) were counted as errors in "normalized" WER. | v1 example #2, #8 | Documented light normalization (adds ة→ه, Arabic-Indic digits, attaching standalone و) + a **space-insensitive CER** to expose word-segmentation errors separately. Raw metrics still reported. |

WER/CER implementation was correct (`jiwer.wer(reference, hypothesis)` order, corpus-level). Harmless warnings: missing `proj_out.weight` (tied to decoder embeddings — verified), attention-mask warning (Whisper uses fixed 30 s inputs; the encoder ignores the mask), "Moving attributes to generation config" (legacy keys in the upstream config), gradio/diffusers vs huggingface-hub (unused packages).

## Model choice (evidence-based)

| Candidate | Params | Verdict |
|---|---|---|
| **A: Whisper-small, full fine-tuning** | 244 M | **Selected.** Fits a 16 GB T4 at batch 8 in fp16 without gradient checkpointing; CT2 int8 ≈ 250 MB, runs faster than real time on GPU and on a 4-core CPU (measured below). Full FT is preferred over LoRA because Derja is a large domain/orthography shift; LoRA on a model this small saves little compute (encoder forward/backward dominates) and usually lags full FT on new dialects. A LoRA pilot is included (`RUN_FT_PILOT`) if you want to verify on your hardware. |
| B: Whisper-base | 74 M | Kept as a measured zero-shot pilot only. ~3× fewer FLOPs, but noticeably weaker Arabic acoustic/lexical prior; published Tunisian fine-tunes of base on HF report >100 % WER. Not worth the accuracy loss since small already meets real-time on CPU int8. |
| C1: `linagora/linto-asr-ar-tn-0.1` (Kaldi TDNN, Vosk) | — | Natively streaming, very low latency, trained by LinTO on these corpora with a 4-gram LM. Not a fine-tuning starting point for Whisper (different architecture), so it is benchmarked as a **low-latency comparison** (`RUN_VOSK_BASELINE`). Its training data overlaps the LinTO train splits; test-split provenance is not stated on the card. |
| C2: `medyas/FarukSTT` (Whisper-large-v3 fine-tune) | 1.55 B | Self-reported ~32 % WER on its *own* validation split — not comparable; training set provenance (54 k pairs, possibly including LinTO/TunSwitch) unknown ⇒ leakage risk; far too heavy for 12 h T4 fine-tuning or CPU real-time. Rejected as default. |
| C3: SalahZa TunSwitch wav2vec2/CTC models (SpeechBrain) | ~300 M | Trained on the very TunSwitch train/test material; would need a different training stack; rejected to keep the pipeline focused. |
| Whisper-large-v3-turbo + LoRA | 809 M | Plausibly more accurate but ~3× slower training on T4 (fewer steps in 12 h) and ~3× slower CPU inference. Possible future experiment, not the default. |

## 0. Dependencies
Run this cell **once per session, before anything imports torch**. It installs only what is missing, pins `torch/torchvision/torchaudio` to the versions Kaggle already has (pip can never replace them), then test-imports torch in a clean subprocess. If you ever see `RpcBackendOptions ... already defined`, the kernel imported torch twice: **Run → Restart kernel**, then run from the configuration cell (no need to reinstall).

In [1]:
import os, sys, time, subprocess
from pathlib import Path
_W = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work"); _W.mkdir(exist_ok=True)
_ss = _W / ".session_start"                     # 12 h budget counts from the first run, survives kernel restarts
if not _ss.exists(): _ss.write_text(str(time.time()))
if "torch" in sys.modules:
    print("⚠ torch is already imported in this kernel. If the next cell fails, restart the kernel first.")
import glob
_torch_dirs = sorted({os.path.realpath(p) for p in glob.glob("/usr/local/lib/python3*/*-packages/torch")
                      + glob.glob("/usr/lib/python3*/*-packages/torch") + glob.glob("/root/.local/lib/python3*/site-packages/torch")
                      + glob.glob("/opt/conda/lib/python3*/site-packages/torch")})
print("torch installations found:", _torch_dirs)
if len(_torch_dirs) > 1:
    print("⚠ More than one torch is installed — this alone causes 'RpcBackendOptions already defined'. "
          "Use Run → Factory reset (fresh environment) or delete the extra copy.")

REQS = ["transformers>=4.50,<5", "datasets>=4.0,<5", "accelerate>=1.3", "jiwer>=3.0", "faster-whisper>=1.1.1",
        "ctranslate2>=4.4", "av<19", "soundfile", "librosa", "psutil"]
_check = (
    "import sys, importlib.metadata as m\n"
    "from packaging.requirements import Requirement\n"
    "for r in sys.argv[1:]:\n"
    "    q = Requirement(r)\n"
    "    try:\n"
    "        if m.version(q.name) not in q.specifier: print(r)\n"
    "    except m.PackageNotFoundError: print(r)\n")
missing = subprocess.run([sys.executable, "-c", _check, *REQS], capture_output=True, text=True).stdout.split()
pins = []
for pkg in ("torch", "torchvision", "torchaudio"):
    try:
        import importlib.metadata as _m; pins.append(f"{pkg}=={_m.version(pkg)}")
    except Exception:
        pass
(_W / "torch-constraints.txt").write_text("\n".join(pins) + "\n")
if missing:
    print("Installing:", missing, "| keeping", pins)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(_W / "torch-constraints.txt"), *missing],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-2500:])
    if r.returncode != 0: raise RuntimeError("pip install failed (see above)")
else:
    print("All requirements already satisfied.")
probe = subprocess.run([sys.executable, "-c", "import torch, transformers, datasets, faster_whisper; "
                        "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), "
                        "'| transformers', transformers.__version__, '| datasets', datasets.__version__)"],
                       capture_output=True, text=True)
print(probe.stdout.strip() or probe.stderr[-3000:])
if probe.returncode != 0:
    raise RuntimeError("torch/transformers do not import cleanly in a fresh interpreter — see the traceback above.")

torch installations found: ['/usr/local/lib/python3.13/dist-packages/torch']
Installing: ['transformers>=4.50,<5', 'jiwer>=3.0', 'faster-whisper>=1.1.1', 'ctranslate2>=4.4', 'av<19'] | keeping ['torch==2.11.0+cu128', 'torchvision==0.26.0+cu128', 'torchaudio==2.11.0+cu128']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.6 MB/s eta 0:00:00
 ERROR: pip's depe

## 1. Configuration — the only cell you normally edit

In [2]:
import os, re, gc, sys, json, math, time, random, hashlib, shutil, unicodedata, collections
from pathlib import Path
_ss = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")) / ".session_start"
SESSION_START = float(_ss.read_text()) if _ss.exists() else time.time()
# A failed `import torch` removes "torch" from sys.modules but leaves its compiled parts loaded;
# any re-import then dies with "RpcBackendOptions ... already defined".
if "torch" not in sys.modules and any(m.startswith("torch.") for m in sys.modules):
    raise RuntimeError("An earlier `import torch` failed in THIS kernel (scroll up for its real error). "
                       "Run → Restart kernel, run the install cell, then this cell. Re-running without a restart can never work.")

RUN_MODE = "full"          # "smoke" (default, ~30 min) | "full" (budgeted to the session)
SMOKE = RUN_MODE == "smoke"
GPU_MODE = "auto"           # "auto": use every visible GPU (DataParallel) | "single": GPU 0 only
if GPU_MODE == "single":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must happen before torch touches CUDA

SESSION_LIMIT_H = 12.0      # Kaggle hard limit
RESERVE_H = 0.5             # extra safety margin on top of the measured final-eval/export estimate
BASE_MODEL = "openai/whisper-small"

AUG_DATASET = "linagora/linto-dataset-audio-ar-tn-augmented"
ORIG_DATASET = "linagora/linto-dataset-audio-ar-tn"
TRAIN_SOURCES = [            # (dataset, config, split, sampling probability)
    (AUG_DATASET, "default", "train", 0.6),
    (ORIG_DATASET, "default", "train", 0.4),
]
EVAL_CONFIGS = None          # None = every original config that has a "test" split
DEV_FRACTION = 0.4           # share of recording groups assigned to dev (rest = final test)
DEV_MAX = 48 if SMOKE else 400
TEST_MAX = 48 if SMOKE else 1200
MAX_SEGS_PER_GROUP_EVAL = 4  # avoid one long recording dominating an eval set

MICRO_BATCH = 8              # per GPU; automatically halved (and accumulation doubled) on OOM
TARGET_BATCH = 32            # effective batch per optimizer step (all GPUs)
EVAL_BATCH = 16
LEARNING_RATE = 1e-5
WARMUP_STEPS = 10 if SMOKE else 300
WEIGHT_DECAY = 0.01
N_EVALS = 3 if SMOKE else 10 # checkpoint evaluations per run
SMOKE_STEPS = 40
FALLBACK_STEPS = 4000        # only used if the throughput benchmark is disabled
BENCH_STEPS = 12 if SMOKE else 30

MIN_SEC, MAX_SEC = 0.6, 29.0
MAX_LABEL_TOKENS = 220
PADDING_SEC = 0.08
MAX_SEGS_PER_ROW_TRAIN = 4   # decorrelate long recordings
RAW_SHUFFLE_BUFFER = 4       # rows held per stream; a row is a whole recording (up to ~14 min) -> keep small
FEATURE_SHUFFLE_BUFFER = 32  # ~1 MB per spectrogram
AUG_PROB = 0.3               # light gain/noise augmentation probability (training only)
SPEC_AUGMENT = not SMOKE

RUN_LEAKAGE_AUDIT = True     # text-only scan of augmented TunSwitch configs vs eval sentences
AUDIT_MAX_ROWS = 3000 if SMOKE else 80000
RUN_PILOT = True             # zero-shot base vs small on a dev subset (+ speed)
PILOT_N = 24 if SMOKE else 120
RUN_FT_PILOT = False         # optional: short full-FT vs LoRA comparison (costs ~1 h in full mode)
RUN_VOSK_BASELINE = not SMOKE
RESUME = "auto"              # "auto" | None | explicit checkpoint path
RESUME_FROM_DIR = None       # e.g. "/kaggle/input/<previous-version>/whisper-tn-v3" in a new session
SEED = 42

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
OUT_DIR = WORK / "whisper-tn-v3"
CT2_DIR = WORK / "whisper-tn-v3-ct2"
CACHE_DIR = WORK / "tn_cache"
STATS_DIR = WORK / "tn_stats"
for d in (OUT_DIR, CACHE_DIR, STATS_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

import numpy as np
import torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
N_GPU = torch.cuda.device_count()
if N_GPU == 0:
    raise RuntimeError("No GPU visible. Enable a GPU accelerator in Kaggle settings.")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = max(1, min(3, CPU_COUNT - 1))   # leave one core (and its RAM) for the main process
def elapsed_h(): return (time.time() - SESSION_START) / 3600
print(f"Mode={RUN_MODE} | GPUs={N_GPU} {GPU_NAMES} | CPUs={CPU_COUNT} | workers={NUM_WORKERS} | elapsed {elapsed_h():.2f} h")
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    if tok: login(token=tok, add_to_git_credential=False); print("HF token supplied")
except Exception:
    print("No HF_TOKEN secret (public data still works).")

Mode=full | GPUs=2 ['Tesla T4', 'Tesla T4'] | CPUs=4 | workers=3 | elapsed 0.01 h
HF token supplied


## 2. Text policies and metrics

* **Training text (`clean_text`)** — the target the model learns. Only lossless clean-up: Unicode NFKC, remove diacritics/tatweel/zero-width marks and TunSwitch `<…>` code-switch tags, collapse spaces. Dialect spelling, French/English words (Latin or Arabic script) and punctuation are kept.
* **Evaluation (`normalized_for_wer`)** — applied identically to reference and hypothesis, only for the *normalized* metrics: lowercase Latin, Arabic-Indic digits → 0-9, أ/إ/آ/ٱ→ا, ى→ي, ة→ه, punctuation/symbols removed, standalone conjunction «و» attached to the next word. These are spelling conventions that Derja lacks rules for; they are not recognition errors.
* **Raw** metrics use `clean_text` only. **Space-insensitive CER** (normalized, spaces deleted) isolates word-segmentation errors (`ما عادش` vs `ماعادش`).
* Corpus-level WER/CER = total edits / total reference units. 95 % CIs via cluster bootstrap over recording groups.

In [3]:
import jiwer
HARAKAT = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭـ]")
ZERO_WIDTH = re.compile(r"[​-‏‪-‮⁦-⁩﻿]")
SPACE = re.compile(r"\s+")
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
LATIN = re.compile(r"[A-Za-zÀ-ÿ]")

def clean_text(text):
    if not isinstance(text, str): return ""
    t = unicodedata.normalize("NFKC", text)
    t = ZERO_WIDTH.sub("", t)
    t = HARAKAT.sub("", t)
    t = t.replace("<", " ").replace(">", " ")
    return SPACE.sub(" ", t).strip()

def normalized_for_wer(text):
    t = clean_text(text).lower().translate(AR_DIGITS)
    t = re.sub(r"[إأآٱ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in t)
    t = SPACE.sub(" ", t).strip()
    t = re.sub(r"(?:(?<=\s)|^)و (?=\S)", "و", t)
    return t

def _edits(ref, hyp, char=False):
    if char:
        o = jiwer.process_characters(ref, hyp)
    else:
        o = jiwer.process_words(ref, hyp)
    return o.substitutions, o.deletions, o.insertions, o.hits + o.substitutions + o.deletions

def per_utterance(refs, hyps):
    rows = []
    for r, h in zip(refs, hyps):
        rr, hh = clean_text(r), clean_text(h)
        rn, hn = normalized_for_wer(r), normalized_for_wer(h)
        row = {}
        for name, a, b, ch in (("raw_w", rr, hh, False), ("raw_c", rr, hh, True),
                               ("norm_w", rn, hn, False), ("norm_c", rn, hn, True),
                               ("nospace_c", rn.replace(" ", ""), hn.replace(" ", ""), True)):
            row[name] = _edits(a, b, ch) if a else (0, 0, len(b.split()) if not ch else len(b), 0)
        rows.append(row)
    return rows

def corpus_rate(rows, key, idx=None):
    idx = range(len(rows)) if idx is None else idx
    S = D = I = N = 0
    for i in idx:
        s, d, ins, n = rows[i][key]; S += s; D += d; I += ins; N += n
    return 100.0 * (S + D + I) / max(N, 1), (100.0 * S / max(N, 1), 100.0 * D / max(N, 1), 100.0 * I / max(N, 1))

def score_corpus(refs, hyps, groups=None, n_boot=1000, seed=SEED):
    rows = per_utterance(refs, hyps)
    out = {"n_utts": len(rows), "n_ref_words": sum(r["raw_w"][3] for r in rows)}
    for key, name in (("raw_w", "wer"), ("raw_c", "cer"), ("norm_w", "norm_wer"),
                      ("norm_c", "norm_cer"), ("nospace_c", "nospace_cer")):
        out[name] = corpus_rate(rows, key)[0]
    _, (s, d, i) = corpus_rate(rows, "norm_w")
    out.update(norm_sub=s, norm_del=d, norm_ins=i)
    if groups is not None and n_boot:
        rng = np.random.default_rng(seed)
        by_g = collections.defaultdict(list)
        for k, g in enumerate(groups): by_g[g].append(k)
        keys = list(by_g)
        for key, name in (("norm_w", "norm_wer"), ("raw_w", "wer")):
            vals = []
            for _ in range(n_boot):
                pick = rng.integers(0, len(keys), len(keys))
                idx = [k for p in pick for k in by_g[keys[p]]]
                vals.append(corpus_rate(rows, key, idx)[0])
            out[name + "_ci95"] = [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]
    return out

# self-tests (no network)
assert clean_text("  أَهْلا   بيك <bonjour> ") == "أهلا بيك bonjour"
assert normalized_for_wer("إنتي، ماشي؟") == "انتي ماشي"
assert normalized_for_wer("و مشيت للمدرسة") == "ومشيت للمدرسه"
assert normalized_for_wer("Bonjour, ça va ٣") == "bonjour ça va 3"
_s = score_corpus(["و مشيت", "a b c"], ["ومشيت", "a x"], groups=["g1", "g2"], n_boot=50)
assert _s["norm_wer"] == 50.0 and _s["wer"] > _s["norm_wer"], _s
print("Text/metric self-tests OK:", {k: round(v, 1) for k, v in _s.items() if isinstance(v, float)})

Text/metric self-tests OK: {'wer': 80.0, 'cer': 36.4, 'norm_wer': 50.0, 'norm_cer': 30.0, 'nospace_cer': 25.0, 'norm_sub': 25.0, 'norm_del': 25.0, 'norm_ins': 0.0}


## 3. Data access, recording groups, segment validation

`source_group(audio_id)` derives a recording/source key (drops `_augmented_` prefixes and `_chunk…`/trailing counters) so dev/test are split **by recording**, never by segment. If a heuristic group swallows >20 % of a sub-corpus, that sub-corpus falls back to one group per row. **Limitation:** the LinTO datasets expose no speaker IDs, so speaker-disjointness cannot be guaranteed; TunSwitch TO has only 89 speakers, so some speaker overlap between train and test is likely.

In [4]:
from datasets import load_dataset, Audio, Dataset, get_dataset_config_names, get_dataset_split_names
from datasets.distributed import split_dataset_by_node
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="arabic", task="transcribe")
extractor, tokenizer = processor.feature_extractor, processor.tokenizer
tokenizer.set_prefix_tokens(language="arabic", task="transcribe", predict_timestamps=False)

def open_split(dataset, config, split):
    # batch_size=1: read parquet one row at a time instead of whole row groups (each row = a full recording)
    ds = load_dataset(dataset, name=config, split=split, streaming=True, batch_size=1)
    return ds.cast_column("audio", Audio(sampling_rate=16000))

def source_group(audio_id):
    s = str(audio_id)
    s = re.sub(r"(?i)^.*?_augmented_", "", s)
    s = re.split(r"(?i)_chunk", s)[0]
    s = re.sub(r"(?:[_-]\d+)+$", "", s)
    return s or str(audio_id)

def stable_hash01(key):
    return int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF

if EVAL_CONFIGS is None:
    try:
        EVAL_CONFIGS = []
        for c in get_dataset_config_names(ORIG_DATASET):
            if c == "default": continue            # default/test = union of the others
            try:
                if "test" in get_dataset_split_names(ORIG_DATASET, c): EVAL_CONFIGS.append(c)
            except Exception as e:
                print("  could not list splits for", c, repr(e)[:80])
    except Exception as e:
        print("Config discovery failed:", repr(e)[:120])
    if not EVAL_CONFIGS: EVAL_CONFIGS = ["TunSwitchCS", "TunSwitchTO"]
print("Evaluation sub-corpora (original, test splits):", EVAL_CONFIGS)

for name, cfg, split, _ in TRAIN_SOURCES:
    r = next(iter(open_split(name, cfg, split)))
    print(f"{name}:{cfg}/{split} columns={list(r)} audio_id={r['audio_id']!r} -> group={source_group(r['audio_id'])!r}")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

linagora/linto-dataset-audio-ar-tn-augmented:default/train columns=['audio_id', 'audio', 'segments', 'transcript'] audio_id='Ba_augmented_1002' -> group='1002'


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

linagora/linto-dataset-audio-ar-tn:default/train columns=['audio_id', 'audio', 'segments', 'transcript'] audio_id='amenyKH_0000001_0000020_1_20' -> group='amenyKH'


In [5]:
class Stats(collections.Counter):
    def report(self, title):
        tot_seg = self["segments_seen"]
        print(f"--- {title}: rows={self['rows_seen']} segments={tot_seg} accepted={self['accepted']} "
              f"({100*self['accepted']/max(tot_seg,1):.1f}%)")
        for k, v in sorted(self.items(), key=lambda kv: -kv[1]):
            if k.startswith("reject:") or k.startswith("row_reject:"):
                print(f"    {k:<38} {v}")

def _audio_duration(audio):
    if isinstance(audio, dict):
        return len(audio["array"]) / audio["sampling_rate"]
    md_ = getattr(audio, "metadata", None)
    for attr in ("duration_seconds", "duration_seconds_from_header"):
        v = getattr(md_, attr, None) if md_ is not None else None
        if v: return float(v)
    return None

def _decode(audio, start, end):
    if isinstance(audio, dict):
        arr, sr = np.asarray(audio["array"], dtype=np.float32), int(audio["sampling_rate"])
        if arr.ndim > 1: arr = arr.mean(axis=0 if arr.shape[0] <= 2 else 1)
        if start is not None:
            arr = arr[max(0, int((start - PADDING_SEC) * sr)):int((end + PADDING_SEC) * sr)]
    else:
        if start is None:
            s = audio.get_all_samples()
        else:
            s = audio.get_samples_played_in_range(max(0.0, start - PADDING_SEC), end + PADDING_SEC)
        arr, sr = s.data, int(s.sample_rate)
        if isinstance(arr, torch.Tensor): arr = arr.detach().cpu().numpy()
        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim > 1: arr = arr.mean(axis=0)
    if sr != 16000:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return np.ascontiguousarray(arr, dtype=np.float32)

def _row_segments(row):
    segs = row.get("segments")
    if isinstance(segs, dict):
        keys = list(segs); n = len(segs[keys[0]]) if keys else 0
        segs = [{k: segs[k][i] for k in keys} for i in range(n)]
    out = []
    if isinstance(segs, (list, tuple)) and segs:
        for seg in segs:
            try: st, en = float(seg["start"]), float(seg["end"])
            except Exception: st = en = float("nan")
            out.append((st, en, clean_text(seg.get("transcript", ""))))
    else:
        out.append((None, None, clean_text(row.get("transcript", ""))))
    return out

def iter_valid_segments(row, stats, max_segs=None, rng=None, blocklist=None):
    # Yields (text, wave, start, end) for segments that pass every check.
    stats["rows_seen"] += 1
    segs = _row_segments(row)
    stats["segments_seen"] += len(segs)
    if blocklist:
        for _, _, t in segs:
            nt = normalized_for_wer(t)
            if len(nt.split()) >= 3 and nt in blocklist:
                stats["row_reject:eval_sentence_overlap"] += 1; stats["reject:in_overlapping_row"] += len(segs)
                return
    dur = None
    try: dur = _audio_duration(row["audio"])
    except Exception: pass
    if dur is not None and any(st is not None and math.isfinite(en) and en > dur + 0.25 for st, en, _ in segs):
        stats["row_reject:timestamps_beyond_audio"] += 1; stats["reject:in_misaligned_row"] += len(segs)
        return
    order = list(range(len(segs)))
    if max_segs is not None and len(order) > max_segs:
        (rng or random).shuffle(order); order = sorted(order[:max_segs])
        stats["reject:per_row_cap"] += len(segs) - max_segs
    for i in order:
        st, en, text = segs[i]
        if st is not None and not (math.isfinite(st) and math.isfinite(en) and en > st >= 0):
            stats["reject:bad_timestamps"] += 1; continue
        if len(text) < 2: stats["reject:empty_text"] += 1; continue
        if not normalized_for_wer(text): stats["reject:no_letters"] += 1; continue
        span = (en - st) if st is not None else None
        if span is not None and not (MIN_SEC <= span <= MAX_SEC):
            stats["reject:duration"] += 1; continue
        if span is not None and not (2.0 <= len(text) / span <= 45.0):
            stats["reject:chars_per_sec"] += 1; continue
        try:
            wave = _decode(row["audio"], st, en)
        except Exception as e:
            stats["reject:decode_error"] += 1
            if stats["reject:decode_error"] <= 3: print("  decode error:", repr(e)[:160])
            continue
        d = len(wave) / 16000
        expected = (min(en + PADDING_SEC, dur or 1e9) - max(0.0, st - PADDING_SEC)) if span is not None else None
        if expected is not None and abs(d - expected) > 0.35:
            stats["reject:decoded_length_mismatch"] += 1; continue
        if not (MIN_SEC <= d <= 29.9):
            stats["reject:duration"] += 1; continue
        if st is None and not (2.0 <= len(text) / d <= 45.0):
            stats["reject:chars_per_sec"] += 1; continue
        if not np.isfinite(wave).all() or float(np.sqrt(np.mean(wave ** 2))) < 1e-4:
            stats["reject:silent_or_corrupt"] += 1; continue
        stats["accepted"] += 1
        yield text, wave, st, en
print("Segment validation ready.")

Segment validation ready.


## 4. Fixed dev / held-out test sets (recording-disjoint, same distribution)

All original `test` splits are read once (≈800 rows), segments validated, grouped by recording and assigned to **dev** (`DEV_FRACTION` of groups) or **test** by a stable hash. Dev is used for checkpoint selection; **test is touched once, after training**. Cached to disk, so re-runs and resumed sessions see identical clips.

In [6]:
def build_eval_pool():
    tag = hashlib.md5(json.dumps([EVAL_CONFIGS, DEV_FRACTION, MAX_SEGS_PER_GROUP_EVAL, MIN_SEC, MAX_SEC, 3]).encode()).hexdigest()[:8]
    path = CACHE_DIR / f"eval_pool_{tag}"
    if path.exists():
        print("Loading cached eval pool:", path); return Dataset.load_from_disk(str(path))
    recs, stats = [], Stats()
    for cfg in EVAL_CONFIGS:
        rows = []
        try:
            for row in open_split(ORIG_DATASET, cfg, "test"):
                rows.append((source_group(row["audio_id"]), row))
        except Exception as e:
            print("  skip", cfg, repr(e)[:120]); continue
        # try recording-level keys from coarse to fine; keep the first that is not degenerate
        cands = [("heuristic", lambda r: source_group(r["audio_id"])),
                 ("first-2-tokens", lambda r: "_".join(str(r["audio_id"]).split("_")[:2])),
                 ("row", lambda r: str(r["audio_id"]))]
        for cname, fn in cands:
            keys = [fn(r) for _, r in rows]
            top = collections.Counter(keys).most_common(1)[0][1] if keys else 0
            if cname == "row" or (top <= max(3, 0.2 * len(rows)) and len(set(keys)) > 1):
                break
        print(f"  {cfg}: grouping by {cname}")
        rows = [(f"{cfg}:{k}", r) for k, (_, r) in zip(keys, rows)]
        per_group = collections.Counter()
        rng = random.Random(SEED)
        for g, row in rows:
            for text, wave, st, en in iter_valid_segments(row, stats, max_segs=MAX_SEGS_PER_GROUP_EVAL, rng=rng):
                if per_group[g] >= MAX_SEGS_PER_GROUP_EVAL: break
                per_group[g] += 1
                recs.append({"audio": wave, "text": text, "group": g, "config": cfg,
                             "audio_id": str(row["audio_id"]), "duration": len(wave) / 16000,
                             "split": "dev" if stable_hash01(g) < DEV_FRACTION else "test"})
        print(f"  {cfg}: {len(rows)} rows, {len(per_group)} groups, {sum(per_group.values())} segments")
    stats.report("eval pool")
    ds = Dataset.from_list(recs)
    ds.save_to_disk(str(path))
    return Dataset.load_from_disk(str(path))

eval_pool = build_eval_pool()
def take(split, n, seed):
    d = eval_pool.filter(lambda s: s == split, input_columns="split")
    return d.shuffle(seed=seed).select(range(min(n, len(d))))
dev_ds = take("dev", DEV_MAX, SEED + 1)
test_ds = take("test", TEST_MAX, SEED + 2)
assert not set(dev_ds["group"]) & set(test_ds["group"]), "dev/test share a recording group!"
for name, d in (("DEV", dev_ds), ("TEST", test_ds)):
    print(f"{name}: {len(d)} utts, {len(set(d['group']))} groups, {sum(d['duration'])/60:.1f} min,",
          dict(collections.Counter(d["config"])))

# Block-list = normalized eval sentences with >=3 words (common 1-2 word phrases would block legit data)
EVAL_BLOCKLIST = {normalized_for_wer(t) for t in eval_pool["text"] if len(normalized_for_wer(t).split()) >= 3}
print("Eval sentences on the training block-list:", len(EVAL_BLOCKLIST))

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

  AmenyKH: grouping by row
  AmenyKH: 44 rows, 44 groups, 44 segments


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

  ApprendreLeTunisien: grouping by heuristic
  ApprendreLeTunisien: 116 rows, 108 groups, 108 segments


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

...

  Youtube_TNScrapped_V1: grouping by heuristic
  Youtube_TNScrapped_V1: 5 rows, 5 groups, 19 segments
--- eval pool: rows=799 segments=1013 accepted=813 (80.3%)
    reject:per_row_cap                     190
    reject:chars_per_sec                   7
    reject:duration                        3


Saving the dataset (0/1 shards):   0%|          | 0/805 [00:00<?, ? examples/s]

Filter:   0%|          | 0/805 [00:00<?, ? examples/s]

Filter:   0%|          | 0/805 [00:00<?, ? examples/s]

DEV: 323 utts, 320 groups, 23.4 min, {'TunSwitchTO': 152, 'TunSwitchCS': 114, 'AmenyKH': 14, 'ApprendreLeTunisien': 39, 'Youtube_TNScrapped_V1': 4}
TEST: 482 utts, 459 groups, 36.6 min, {'TunSwitchTO': 171, 'TunSwitchCS': 185, 'ApprendreLeTunisien': 69, 'AmenyKH': 30, 'Youtube_TNScrapped_V1': 15, 'OneStory': 12}
Eval sentences on the training block-list: 679


In [7]:
# Optional up-front leakage audit: text-only scan of the augmented TunSwitch configs.
leak_audit = {}
if RUN_LEAKAGE_AUDIT:
    t0 = time.time()
    try:
        aug_cfgs = [c for c in get_dataset_config_names(AUG_DATASET) if "tunswitch" in c.lower()]
    except Exception:
        aug_cfgs = ["TunSwitchCS_VCA", "TunSwitchTO_VCA"]
    for cfg in aug_cfgs:
        seen = hit = 0; hit_texts = set()
        try:
            try:   # column projection avoids downloading audio bytes
                ds = load_dataset(AUG_DATASET, name=cfg, split="train", streaming=True, columns=["audio_id", "transcript"])
            except Exception:
                ds = load_dataset(AUG_DATASET, name=cfg, split="train", streaming=True).remove_columns(["audio"])
            for r in ds:
                seen += 1
                nt = normalized_for_wer(r["transcript"])
                if nt in EVAL_BLOCKLIST: hit += 1; hit_texts.add(nt)
                if seen >= AUDIT_MAX_ROWS or time.time() - t0 > (300 if SMOKE else 1200): break
        except Exception as e:
            print("  audit failed for", cfg, repr(e)[:120])
        leak_audit[cfg] = {"rows_scanned": seen, "rows_matching_eval_sentence": hit,
                           "distinct_eval_sentences_found": len(hit_texts)}
        print(f"  {cfg}: scanned {seen} rows, {hit} contain an eval sentence "
              f"({len(hit_texts)}/{len(EVAL_BLOCKLIST)} distinct eval sentences)")
    print(f"Audit took {time.time()-t0:.0f}s. Matching rows are excluded from training by the block-list.")

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

  TunSwitchCS_VCA: scanned 37639 rows, 14 contain an eval sentence (1/679 distinct eval sentences)


Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

  TunSwitchTO_VCA: scanned 15365 rows, 7 contain an eval sentence (1/679 distinct eval sentences)
Audit took 288s. Matching rows are excluded from training by the block-list.


## 5. Training stream (shard-parallel, mixed sources, streaming — no full download)

Each DataLoader worker reads *different* parquet shards (`split_dataset_by_node`), shuffles shard order and rows, interleaves the augmented and original corpora by probability, keeps ≤4 segments per recording, applies light gain/noise augmentation, computes log-Mel features and shuffles them in a buffer. Rejection statistics from every worker are written to `tn_stats/`.

In [8]:
from torch.utils.data import IterableDataset as TorchIterable, get_worker_info

def augment(wave, rng):
    if rng.random() >= AUG_PROB: return wave
    w = wave * (10 ** (rng.uniform(-6, 6) / 20))
    if rng.random() < 0.5:
        snr = rng.uniform(15, 30)
        p = float(np.mean(w ** 2)) + 1e-10
        noise = rng.standard_normal(len(w)).astype(np.float32)
        noise = np.convolve(noise, np.ones(4) / 4, mode="same")            # slightly coloured
        w = w + noise * math.sqrt(p / (10 ** (snr / 10)) / (float(np.mean(noise ** 2)) + 1e-10))
    peak = float(np.max(np.abs(w))) + 1e-9
    return (w / peak * 0.99).astype(np.float32) if peak > 1 else w.astype(np.float32)

class StreamingTrainSet(TorchIterable):
    def __init__(self, sources, seed, blocklist):
        self.sources, self.seed, self.blocklist = sources, seed, blocklist
    def _source_iter(self, k, wid, nw, rng, stats):
        name, cfg, split, _ = self.sources[k]
        epoch = 0
        while True:
            raw = open_split(name, cfg, split)
            if nw > 1:   # disjoint parquet shards per worker (no duplicated downloads)
                raw = raw.shard(num_shards=nw, index=wid) if raw.num_shards >= nw else split_dataset_by_node(raw, rank=wid, world_size=nw)
            raw = raw.shuffle(seed=self.seed + 7919 * epoch + 31 * wid + k, buffer_size=RAW_SHUFFLE_BUFFER)
            try:
                for row in raw:
                    for text, wave, _, _ in iter_valid_segments(row, stats, MAX_SEGS_PER_ROW_TRAIN, rng, self.blocklist):
                        ids = tokenizer(text).input_ids
                        if len(ids) > MAX_LABEL_TOKENS: stats["reject:too_many_tokens"] += 1; continue
                        feats = extractor(augment(wave, np.random.default_rng(rng.getrandbits(32))), sampling_rate=16000).input_features[0]
                        yield {"input_features": feats.astype(np.float32), "labels": ids}
                stats[f"epoch_wrap:{cfg}"] += 1
            except Exception as e:   # transient HTTP/decode failure: log, back off, reopen the stream reshuffled
                stats[f"stream_error:{cfg}"] += 1
                print(f"[worker {wid}] stream error on {cfg}: {repr(e)[:160]} — reopening", flush=True)
                time.sleep(min(60, 5 * stats[f"stream_error:{cfg}"]))
            epoch += 1
    def __iter__(self):
        info = get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        rng = random.Random(self.seed * 1000 + wid)
        stats = Stats()
        its = [self._source_iter(k, wid, nw, rng, stats) for k in range(len(self.sources))]
        probs = np.array([s[3] for s in self.sources], dtype=float); probs /= probs.sum()
        buf, n = [], 0
        while True:
            k = int(np.searchsorted(np.cumsum(probs), rng.random()))
            buf.append(next(its[k])); n += 1
            if n % 100 == 0:
                (STATS_DIR / f"worker{wid}_seed{self.seed}.json").write_text(json.dumps(stats))
            if len(buf) >= FEATURE_SHUFFLE_BUFFER:
                yield buf.pop(rng.randrange(len(buf)))

def training_stats():
    tot = Stats()
    for f in STATS_DIR.glob("worker*.json"): tot.update(json.loads(f.read_text()))
    return tot

_probe = next(iter(StreamingTrainSet(TRAIN_SOURCES, SEED, EVAL_BLOCKLIST)))
print("Train feature:", _probe["input_features"].shape, "| label tokens:", len(_probe["labels"]))

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Train feature: (80, 3000) | label tokens: 30


## 6. Model, collator, transcription helpers

In [9]:
from dataclasses import dataclass
from transformers import WhisperForConditionalGeneration

import inspect
from transformers.modeling_outputs import Seq2SeqLMOutput

class LossOnlyWhisper(WhisperForConditionalGeneration):
    # In training, return only the loss. Otherwise DataParallel gathers every GPU's full
    # logits (batch x tokens x 51865) onto GPU 0 -> the OOM seen at step 217 in v3.
    def forward(self, *args, **kwargs):
        out = super().forward(*args, **kwargs)
        if self.training and out.loss is not None:
            return Seq2SeqLMOutput(loss=out.loss)
        return out
LossOnlyWhisper.forward.__signature__ = inspect.signature(WhisperForConditionalGeneration.forward)

def fix_architectures(path):
    # keep saved configs loadable by ct2-transformers-converter / plain transformers
    p = Path(path) / "config.json"
    if p.exists():
        c = json.loads(p.read_text()); c["architectures"] = ["WhisperForConditionalGeneration"]
        p.write_text(json.dumps(c, indent=2))

def load_model(name=BASE_MODEL):
    m = LossOnlyWhisper.from_pretrained(name)
    m.config.forced_decoder_ids = None
    m.generation_config.forced_decoder_ids = None
    m.generation_config.language = "arabic"
    m.generation_config.task = "transcribe"
    m.config.use_cache = False
    m.config.apply_spec_augment = SPEC_AUGMENT
    m.config.mask_time_prob, m.config.mask_time_length = 0.05, 10
    m.config.mask_feature_prob, m.config.mask_feature_length = 0.0, 10
    return m

@dataclass
class Collator:
    def __call__(self, feats):
        # per item: training items carry precomputed features, eval items carry raw audio + text
        x = [{"input_features": f["input_features"] if "input_features" in f else
              extractor(np.asarray(f["audio"], np.float32), sampling_rate=16000).input_features[0]} for f in feats]
        batch = extractor.pad(x, return_tensors="pt")
        labels = [f["labels"] if "labels" in f else tokenizer(f["text"]).input_ids[:MAX_LABEL_TOKENS + 4] for f in feats]
        lab = tokenizer.pad([{"input_ids": l} for l in labels], return_tensors="pt")
        lab_ids = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        sot = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        if (lab_ids[:, 0] == sot).all(): lab_ids = lab_ids[:, 1:]    # model prepends it itself
        batch["labels"] = lab_ids
        return batch
collator = Collator()

def compute_metrics(pred):
    p = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    lab = np.array(pred.label_ids, copy=True); lab[lab == -100] = tokenizer.pad_token_id
    p = np.where(p < 0, tokenizer.pad_token_id, p)
    hyps = tokenizer.batch_decode(p, skip_special_tokens=True)
    refs = tokenizer.batch_decode(lab, skip_special_tokens=True)
    s = score_corpus(refs, hyps, groups=None, n_boot=0)
    return {k: s[k] for k in ("wer", "cer", "norm_wer", "norm_cer", "nospace_cer")}

@torch.no_grad()
def transcribe_hf(model, audios, batch_size=EVAL_BATCH, num_beams=1):
    model.eval(); dev = next(model.parameters()).device; dt = next(model.parameters()).dtype
    texts = []; torch.cuda.synchronize(); t0 = time.perf_counter()
    for i in range(0, len(audios), batch_size):
        chunk = [np.asarray(a, np.float32) for a in audios[i:i + batch_size]]
        f = extractor(chunk, sampling_rate=16000, return_tensors="pt").input_features.to(dev, dt)
        out = model.generate(input_features=f, language="arabic", task="transcribe",
                             max_new_tokens=MAX_LABEL_TOKENS, num_beams=num_beams)
        texts += [t.strip() for t in tokenizer.batch_decode(out, skip_special_tokens=True)]
    torch.cuda.synchronize()
    return texts, time.perf_counter() - t0

def audios_of(ds):
    # list of float32 arrays without materialising Python float lists
    return [np.asarray(r["audio"], dtype=np.float32) for r in ds.with_format("numpy")]

def transcribe_ct2(fw, audios, beam_size=1):
    texts = []; t0 = time.perf_counter()
    for a in audios:
        segs, _ = fw.transcribe(np.asarray(a, np.float32), language="ar", task="transcribe", beam_size=beam_size,
                                condition_on_previous_text=False, vad_filter=False, without_timestamps=True)
        texts.append(" ".join(s.text.strip() for s in segs).strip())
    return texts, time.perf_counter() - t0

print("Collator check:", {k: tuple(v.shape) for k, v in collator([_probe, dev_ds[0]]).items()})

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Collator check: {'input_features': (2, 80, 3000), 'labels': (2, 29)}


## 7. Pilot experiments (short, controlled)
Zero-shot accuracy and speed of the candidate starting points on the same dev subset. Zero-shot WER does not predict fine-tuned WER exactly, but a model that is both much slower and not better is eliminated cheaply. `RUN_FT_PILOT` adds two short fine-tuning runs (full vs LoRA, same steps) — off by default to protect the training budget.

In [10]:
pilot = {}
if RUN_PILOT:
    sub = dev_ds.select(range(min(PILOT_N, len(dev_ds))))
    audio_s = sum(sub["duration"])
    for name in ["openai/whisper-base", BASE_MODEL]:
        m = load_model(name).half().cuda()
        hyps, secs = transcribe_hf(m, audios_of(sub))
        s = score_corpus(sub["text"], hyps, n_boot=0)
        pilot[name] = {"norm_wer": s["norm_wer"], "norm_cer": s["norm_cer"], "rtf_hf_fp16_gpu": secs / audio_s,
                       "params_M": sum(p.numel() for p in m.parameters()) / 1e6}
        print(name, {k: round(v, 3) for k, v in pilot[name].items()})
        del m; gc.collect(); torch.cuda.empty_cache()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


openai/whisper-base {'norm_wer': 128.38, 'norm_cer': 79.911, 'rtf_hf_fp16_gpu': 0.029, 'params_M': 72.594}


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

openai/whisper-small {'norm_wer': 112.534, 'norm_cer': 73.619, 'rtf_hf_fp16_gpu': 0.039, 'params_M': 241.735}


In [11]:
ft_pilot = {}
if RUN_FT_PILOT:
    %pip install -q "peft>=0.11"
    from peft import LoraConfig, get_peft_model
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
    PILOT_STEPS = 30 if SMOKE else 300
    sub = dev_ds.select(range(min(PILOT_N, len(dev_ds))))
    for variant in ("full", "lora"):
        m = load_model()
        lr = LEARNING_RATE
        if variant == "lora":
            m = get_peft_model(m, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05,
                                             target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]))
            lr = 1e-3
        a = Seq2SeqTrainingArguments(output_dir=str(WORK / f"pilot_{variant}"), per_device_train_batch_size=MICRO_BATCH,
                                     gradient_accumulation_steps=1, learning_rate=lr, warmup_steps=30, max_steps=PILOT_STEPS,
                                     fp16=True, save_strategy="no", eval_strategy="no", report_to="none",
                                     remove_unused_columns=False, label_names=["labels"], dataloader_num_workers=NUM_WORKERS,
                                     logging_steps=50, seed=SEED)
        tr = Seq2SeqTrainer(model=m, args=a, train_dataset=StreamingTrainSet(TRAIN_SOURCES, SEED + 500, EVAL_BLOCKLIST),
                            data_collator=collator)
        t0 = time.time(); tr.train(); train_s = time.time() - t0
        mm = (m.merge_and_unload() if variant == "lora" else m).half().cuda()
        hyps, _ = transcribe_hf(mm, audios_of(sub))
        ft_pilot[variant] = {"norm_wer": score_corpus(sub["text"], hyps, n_boot=0)["norm_wer"], "s_per_step": train_s / PILOT_STEPS,
                             "peak_gpu_GB": torch.cuda.max_memory_allocated() / 1e9}
        print(variant, ft_pilot[variant])
        del m, mm, tr; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 8. Measured throughput → training budget
A short benchmark (fresh model, real streaming data, OOM fallback) measures seconds per optimizer step and dev-evaluation time. The step limit is then set so that training + periodic evaluation + final evaluation/export fit into the session with `RESERVE_H` to spare. A wall-clock callback also stops training (with a checkpoint) if the budget is exceeded anyway.

In [12]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

class StepTimer(TrainerCallback):
    def __init__(self, skip=5): self.skip, self.times, self._t = skip, [], None
    def on_step_begin(self, args, state, control, **kw): self._t = time.perf_counter()
    def on_step_end(self, args, state, control, **kw):
        if state.global_step > self.skip: self.times.append(time.perf_counter() - self._t)

def make_args(out, max_steps, micro, accum, eval_steps=None, **extra):
    return Seq2SeqTrainingArguments(
        output_dir=str(out), per_device_train_batch_size=micro, per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=accum, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
        warmup_steps=min(WARMUP_STEPS, max(1, max_steps // 10)), weight_decay=WEIGHT_DECAY, max_grad_norm=1.0,
        max_steps=max_steps, fp16=True, optim="adamw_torch_fused", gradient_checkpointing=False,
        eval_strategy="steps" if eval_steps else "no", eval_steps=eval_steps,
        save_strategy="steps" if eval_steps else "no", save_steps=eval_steps, save_total_limit=2,
        logging_steps=10 if SMOKE else 25, predict_with_generate=True, generation_max_length=MAX_LABEL_TOKENS + 8,
        metric_for_best_model="norm_wer", greater_is_better=False, load_best_model_at_end=bool(eval_steps),
        remove_unused_columns=False, label_names=["labels"], dataloader_num_workers=NUM_WORKERS,
        dataloader_prefetch_factor=2, dataloader_persistent_workers=False, report_to="none",
        seed=SEED, data_seed=SEED, ignore_data_skip=True, **extra)

plan_path = OUT_DIR / "run_plan.json"
resume_ckpt = None
if RESUME:
    src = Path(RESUME_FROM_DIR) if RESUME_FROM_DIR else OUT_DIR
    if src != OUT_DIR and src.exists():
        shutil.copytree(src, OUT_DIR, dirs_exist_ok=True)
    resume_ckpt = RESUME if RESUME != "auto" else get_last_checkpoint(str(OUT_DIR))
old_plan = json.loads(plan_path.read_text()) if plan_path.exists() else None
if old_plan is not None and old_plan.get("mode") != RUN_MODE:
    # e.g. leftovers of a smoke run in the same session: never resume those in full mode
    print(f"Found a '{old_plan.get('mode', 'smoke')}' run in {OUT_DIR}; starting a fresh '{RUN_MODE}' run instead of resuming it.")
    shutil.rmtree(OUT_DIR, ignore_errors=True); OUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(STATS_DIR, ignore_errors=True); STATS_DIR.mkdir(parents=True, exist_ok=True)
    resume_ckpt, old_plan = None, None
if resume_ckpt and old_plan is not None:
    plan = old_plan
    plan["resumes"] = plan.get("resumes", 0) + 1
    print("Resuming from", resume_ckpt, "with saved plan", plan)
else:
    resume_ckpt = None
    shutil.rmtree(STATS_DIR, ignore_errors=True); STATS_DIR.mkdir(parents=True, exist_ok=True)
    micro = MICRO_BATCH
    while True:
        accum = max(1, TARGET_BATCH // (micro * N_GPU))
        timer = StepTimer(skip=4)
        m = tr = None
        m = load_model()
        try:
            tr = Seq2SeqTrainer(model=m, args=make_args(WORK / "bench", BENCH_STEPS, micro, accum),
                                train_dataset=StreamingTrainSet(TRAIN_SOURCES, SEED + 999, EVAL_BLOCKLIST),
                                data_collator=collator, callbacks=[timer])
            tr.train()
            peak = torch.cuda.max_memory_allocated() / 1e9
            # time a dev evaluation (generation) on a small subset with the same trainer
            sub = dev_ds.select(range(min(32, len(dev_ds))))
            t0 = time.time(); tr.evaluate(eval_dataset=sub, metric_key_prefix="bench"); eval_per_utt = (time.time() - t0) / len(sub)
            break
        except torch.cuda.OutOfMemoryError:
            print(f"OOM at micro-batch {micro}; halving.")
            micro //= 2
            if micro == 0: raise
        finally:
            tr = m = None; gc.collect(); torch.cuda.empty_cache()
    step_s = float(np.median(timer.times))
    eval_s = eval_per_utt * len(dev_ds)
    # final phase: ~6 test passes (2 HF + 4 CT2) + exports + streaming bench + zip
    final_phase_s = 6 * eval_per_utt * len(test_ds) * 1.3 + 35 * 60
    if RUN_VOSK_BASELINE: final_phase_s += 15 * 60
    avail_s = (SESSION_LIMIT_H - RESERVE_H - elapsed_h()) * 3600 - final_phase_s
    if SMOKE:
        steps = SMOKE_STEPS
    else:
        steps = int((avail_s - N_EVALS * (eval_s + 60)) / step_s)
        steps = max(200, steps)
    eval_steps = max(10, steps // N_EVALS)
    plan = dict(mode=RUN_MODE, micro=micro, accum=accum, n_gpu=N_GPU, effective_batch=micro * accum * N_GPU, max_steps=steps,
                eval_steps=eval_steps, step_s=step_s, eval_s=eval_s, final_phase_s=final_phase_s,
                peak_train_gpu_GB=peak, planned_at_h=elapsed_h(), resumes=0,
                deadline_h=SESSION_LIMIT_H - RESERVE_H - final_phase_s / 3600)
    plan_path.write_text(json.dumps(plan, indent=2))
    print(json.dumps(plan, indent=2))
    print(f"→ ~{steps * plan['effective_batch']:,} training segments, "
          f"estimated training wall time {(steps * step_s + N_EVALS * eval_s) / 3600:.2f} h")
shutil.rmtree(WORK / "bench", ignore_errors=True)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
25,1.558400


{
  "mode": "full",
  "micro": 8,
  "accum": 2,
  "n_gpu": 2,
  "effective_batch": 32,
  "max_steps": 4272,
  "eval_steps": 427,
  "step_s": 8.443101983500014,
  "eval_s": 43.68946638703346,
  "final_phase_s": 3508.5291573643685,
  "peak_train_gpu_GB": 13.33881344,
  "planned_at_h": 0.21761448754204643,
  "resumes": 0,
  "deadline_h": 10.525408567398786
}
→ ~136,704 training segments, estimated training wall time 10.14 h


## 9. Fine-tune (checkpoints every `eval_steps`, best checkpoint by dev normalized WER)

In [13]:
import psutil
class RamMonitor(TrainerCallback):
    # logs RAM of the main process + DataLoader workers; checkpoints early if the machine gets close to OOM
    def __init__(self, every=50, min_free_gb=4.0): self.every, self.min_free_gb, self.saved_low = every, min_free_gb, False
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % self.every: return control
        me = psutil.Process()
        rss = me.memory_info().rss + sum(c.memory_info().rss for c in me.children(recursive=True) if c.is_running())
        free = psutil.virtual_memory().available / 1e9
        print(f"[ram] step {state.global_step}: processes {rss/1e9:.1f} GB, machine free {free:.1f} GB", flush=True)
        if free < self.min_free_gb and not self.saved_low:
            print("[ram] low memory — saving a checkpoint now so a crash loses little", flush=True)
            control.should_save = True; self.saved_low = True
        return control

class TimeBudget(TrainerCallback):
    def __init__(self, deadline_h): self.deadline_h = deadline_h
    def on_step_end(self, args, state, control, **kw):
        if elapsed_h() > self.deadline_h:
            print(f"\n⏱ Time budget reached at step {state.global_step}; saving and stopping.")
            control.should_save = True; control.should_evaluate = True; control.should_training_stop = True
        return control

def build_trainer():
    global model
    model = load_model()
    train_set = StreamingTrainSet(TRAIN_SOURCES, SEED + 100 * plan.get("resumes", 0), EVAL_BLOCKLIST)  # new data order on resume
    return Seq2SeqTrainer(model=model, args=make_args(OUT_DIR, plan["max_steps"], plan["micro"], plan["accum"], plan["eval_steps"]),
                          train_dataset=train_set, eval_dataset=dev_ds, data_collator=collator,
                          compute_metrics=compute_metrics, processing_class=extractor,
                          callbacks=[TimeBudget(plan["deadline_h"]), RamMonitor()])

t_train = time.time()
while True:
    trainer = build_trainer()
    try:
        train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
        break
    except RuntimeError as e:
        if "DataLoader worker" not in str(e) and not isinstance(e, torch.cuda.OutOfMemoryError): raise
        if not isinstance(e, torch.cuda.OutOfMemoryError):      # system-RAM kill of a data worker
            trainer = model = None; gc.collect(); torch.cuda.empty_cache()
            NUM_WORKERS = max(2, NUM_WORKERS - 1)
            resume_ckpt = get_last_checkpoint(str(OUT_DIR))
            print(f"\n⚠ Data worker killed (system RAM). Resuming from {resume_ckpt or 'scratch'} "
                  f"with {NUM_WORKERS} workers", flush=True)
            continue
        if plan["micro"] <= 1: raise
        trainer = model = None; gc.collect(); torch.cuda.empty_cache()
        plan["micro"] //= 2; plan["accum"] *= 2; plan_path.write_text(json.dumps(plan, indent=2))
        resume_ckpt = get_last_checkpoint(str(OUT_DIR))
        print(f"\n⚠ CUDA OOM — retrying with micro-batch {plan['micro']} x accum {plan['accum']} "
              f"(same effective batch), resuming from {resume_ckpt or 'scratch'}", flush=True)
train_hours = (time.time() - t_train) / 3600
best_ckpt = trainer.state.best_model_checkpoint
history = [h for h in trainer.state.log_history if "eval_norm_wer" in h]
print("Best checkpoint:", best_ckpt)
for h in history: print(f"  step {h['step']:>5}: dev norm WER {h['eval_norm_wer']:.2f}  CER {h['eval_norm_cer']:.2f}  loss {h['eval_loss']:.3f}")
trainer.save_model(str(OUT_DIR)); processor.save_pretrained(str(OUT_DIR))
fix_architectures(OUT_DIR)
for c in OUT_DIR.glob("checkpoint-*"): fix_architectures(c)
tstats = training_stats(); tstats.report("training stream (all workers)")
print(f"Training wall time {train_hours:.2f} h | session elapsed {elapsed_h():.2f} h")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer,Cer,Norm Wer,Norm Cer,Nospace Cer
427,0.892700,0.699172,51.038007,21.235210,47.774481,19.942557,20.096197
854,0.744100,0.616842,52.634941,24.815710,51.236400,24.094655,23.974147
1281,0.670300,0.554921,45.033536,18.583906,43.026706,17.738511,17.773937
1708,0.662500,0.531756,40.146918,15.988354,38.608638,15.347153,15.519315
2135,0.627600,0.501680,46.662408,20.213095,45.565447,19.674076,19.149256
2562,0.633900,0.514192,51.676781,29.635136,50.148368,29.020979,29.813618
2989,0.539900,0.505025,44.203130,22.672366,42.697000,22.059191,23.282730
3416,0.541300,0.458402,36.442031,14.997212,35.080778,14.441808,14.782805
3843,0.412900,0.458746,36.953050,15.375085,35.575338,14.766484,15.090936
4071,0.528100,0.456639,36.697541,15.189246,35.311573,14.597902,14.940628


[ram] step 50: processes 24.0 GB, machine free 16.9 GB
[ram] step 100: processes 25.0 GB, machine free 16.3 GB
[ram] step 150: processes 26.7 GB, machine free 13.5 GB
[ram] step 200: processes 29.3 GB, machine free 10.7 GB
[ram] step 250: processes 28.2 GB, machine free 11.3 GB
[ram] step 300: processes 27.1 GB, machine free 13.1 GB
[ram] step 350: processes 27.8 GB, machine free 11.8 GB
[ram] step 400: processes 28.8 GB, machine free 10.8 GB


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension

[ram] step 450: processes 31.0 GB, machine free 8.8 GB
[ram] step 500: processes 32.8 GB, machine free 7.5 GB
[ram] step 550: processes 29.7 GB, machine free 10.2 GB
[ram] step 600: processes 30.6 GB, machine free 10.3 GB


...

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/243 [00:00<?, ?it/s]


⏱ Time budget reached at step 4071; saving and stopping.


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Best checkpoint: /kaggle/working/whisper-tn-v3/checkpoint-3416
  step   427: dev norm WER 47.77  CER 19.94  loss 0.699
  step   854: dev norm WER 51.24  CER 24.09  loss 0.617
  step  1281: dev norm WER 43.03  CER 17.74  loss 0.555
  step  1708: dev norm WER 38.61  CER 15.35  loss 0.532
  step  2135: dev norm WER 45.57  CER 19.67  loss 0.502
  step  2562: dev norm WER 50.15  CER 29.02  loss 0.514
  step  2989: dev norm WER 42.70  CER 22.06  loss 0.505
  step  3416: dev norm WER 35.08  CER 14.44  loss 0.458
  step  3843: dev norm WER 35.58  CER 14.77  loss 0.459
  step  4071: dev norm WER 35.31  CER 14.60  loss 0.457
--- training stream (all workers): rows=81638 segments=4557380 accepted=131204 (2.9%)
    reject:per_row_cap                     4411619
    reject:in_misaligned_row               8407
    reject:duration                        3860
    reject:chars_per_sec                   1766
    reject:in_overlapping_row              463
    row_reject:eval_sentence_overlap       97
   

## 10. Export CTranslate2 (int8_float16 for GPU, loadable as int8 on CPU)

In [14]:
import subprocess
del trainer, model; gc.collect(); torch.cuda.empty_cache()
chk = WhisperForConditionalGeneration.from_pretrained(str(OUT_DIR)); chk.tie_weights()
print("Tied output projection:", chk.proj_out.weight.data_ptr() == chk.model.decoder.embed_tokens.weight.data_ptr())
del chk
cmd = ["ct2-transformers-converter", "--model", str(OUT_DIR), "--output_dir", str(CT2_DIR),
       "--copy_files", "tokenizer.json", "preprocessor_config.json", "--quantization", "int8_float16", "--force"]
subprocess.run(cmd, check=True)
def dir_mb(p):
    p = Path(p)
    return (p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())) / 1e6
print(f"CT2 model: {dir_mb(CT2_DIR):.0f} MB | HF fp32 weights: {dir_mb(OUT_DIR / 'model.safetensors') if (OUT_DIR/'model.safetensors').exists() else float('nan'):.0f} MB")

Tied output projection: True
CT2 model: 250 MB | HF fp32 weights: 967 MB


## 11. One-time held-out test evaluation (identical clips, normalization and greedy decoding for every model)
The test set has not been used for any decision. Beam-3 CT2 is reported separately (it is the deployed "final" decode).

In [15]:
import psutil
from faster_whisper import WhisperModel
T_AUDIO = audios_of(test_ds); T_REF = test_ds["text"]; T_GROUP = test_ds["group"]; T_CFG = test_ds["config"]
audio_s = float(sum(test_ds["duration"]))
results, hyps_by_model = {}, {}

def record(name, hyps, secs, extra=None):
    n = len(hyps)                                   # a model may be scored on the first n test clips only
    refs, groups, cfgs = T_REF[:n], T_GROUP[:n], T_CFG[:n]
    s = score_corpus(refs, hyps, groups=groups)
    s["rtf"] = secs / float(sum(test_ds["duration"][:n])); s.update(extra or {})
    s["runaway_outputs"] = sum(len(normalized_for_wer(h).split()) > 2 * len(normalized_for_wer(r).split()) + 3
                               for r, h in zip(refs, hyps))
    s["per_config_norm_wer"] = {c: score_corpus([r for r, k in zip(refs, cfgs) if k == c],
                                                [h for h, k in zip(hyps, cfgs) if k == c], n_boot=0)["norm_wer"]
                                for c in sorted(set(cfgs))}
    results[name] = s; hyps_by_model[name] = hyps
    print(f"{name:<34} n={n} WER {s['wer']:.1f}  normWER {s['norm_wer']:.1f} {np.round(s['norm_wer_ci95'],1)}  "
          f"CER {s['cer']:.1f}  normCER {s['norm_cer']:.1f}  RTF {s['rtf']:.3f}  runaway {s['runaway_outputs']}")

for name, path in (("whisper-small (base, HF fp16 greedy)", BASE_MODEL), ("fine-tuned (HF fp16 greedy)", str(OUT_DIR))):
    m = WhisperForConditionalGeneration.from_pretrained(path).half().cuda()
    torch.cuda.reset_peak_memory_stats()
    h, secs = transcribe_hf(m, T_AUDIO)
    record(name, h, secs, {"peak_gpu_GB": torch.cuda.max_memory_allocated() / 1e9})
    del m; gc.collect(); torch.cuda.empty_cache()

# Same decoder for both models (CT2 int8_fp16, temperature fallback on loops) = the fair comparison.
BASE_CT2 = WORK / "whisper-small-base-ct2"
if not (BASE_CT2 / "model.bin").exists():
    subprocess.run(["ct2-transformers-converter", "--model", BASE_MODEL, "--output_dir", str(BASE_CT2),
                    "--copy_files", "tokenizer.json", "preprocessor_config.json", "--quantization", "int8_float16", "--force"], check=True)
for label, path in (("whisper-small (base)", BASE_CT2), ("fine-tuned", CT2_DIR)):
    fw = WhisperModel(str(path), device="cuda", compute_type="int8_float16")
    for beam in (1, 3):
        h, secs = transcribe_ct2(fw, T_AUDIO, beam)
        record(f"{label} CT2 int8_fp16 GPU beam{beam}", h, secs)
    del fw; gc.collect()
print("Note: HF rows use plain greedy without temperature fallback, so a looping output is not retried; "
      "compare models on the CT2 rows.")

# CPU deployment check on a subset (4 threads, int8)
n_cpu = min(80 if not SMOKE else 12, len(test_ds))
fw_cpu = WhisperModel(str(CT2_DIR), device="cpu", compute_type="int8", cpu_threads=min(4, CPU_COUNT))
rss0 = psutil.Process().memory_info().rss
h, secs = transcribe_ct2(fw_cpu, T_AUDIO[:n_cpu], 1)
cpu_s = score_corpus(T_REF[:n_cpu], h, n_boot=0)
results["fine-tuned CT2 int8 CPU beam1 (subset)"] = {"n_utts": n_cpu, "norm_wer": cpu_s["norm_wer"], "wer": cpu_s["wer"],
    "cer": cpu_s["cer"], "norm_cer": cpu_s["norm_cer"], "rtf": secs / float(sum(test_ds["duration"][:n_cpu])),
    "cpu_threads": min(4, CPU_COUNT), "rss_delta_MB": (psutil.Process().memory_info().rss - rss0) / 1e6}
print("CPU int8 subset:", {k: round(v, 3) if isinstance(v, float) else v for k, v in results["fine-tuned CT2 int8 CPU beam1 (subset)"].items()})

whisper-small (base, HF fp16 greedy) n=482 WER 96.3  normWER 94.4 [ 85.4 103.6]  CER 63.7  normCER 62.4  RTF 0.037  runaway 12


`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProce

fine-tuned (HF fp16 greedy)        n=482 WER 32.8  normWER 31.9 [29.9 34.2]  CER 13.4  normCER 12.9  RTF 0.025  runaway 0
whisper-small (base) CT2 int8_fp16 GPU beam1 n=482 WER 80.9  normWER 78.1 [75.5 80.9]  CER 46.5  normCER 44.9  RTF 0.092  runaway 0
whisper-small (base) CT2 int8_fp16 GPU beam3 n=482 WER 79.1  normWER 76.3 [73.8 79. ]  CER 44.5  normCER 42.9  RTF 0.078  runaway 0
fine-tuned CT2 int8_fp16 GPU beam1 n=482 WER 34.6  normWER 33.8 [30.5 38.6]  CER 15.0  normCER 14.6  RTF 0.042  runaway 1
fine-tuned CT2 int8_fp16 GPU beam3 n=482 WER 30.4  normWER 29.5 [27.5 31.6]  CER 12.0  normCER 11.5  RTF 0.044  runaway 0
Note: HF rows use plain greedy without temperature fallback, so a looping output is not retried; compare models on the CT2 rows.
CPU int8 subset: {'n_utts': 80, 'norm_wer': 30.33, 'wer': 31.017, 'cer': 13.238, 'norm_cer': 12.854, 'rtf': 0.889, 'cpu_threads': 4, 'rss_delta_MB': 62.202}


In [16]:
# Optional low-latency comparison: LinTO Kaldi/Vosk Tunisian model (native streaming)
if RUN_VOSK_BASELINE:
    try:
        %pip install -q vosk
        import zipfile, vosk
        from huggingface_hub import hf_hub_download
        vz = hf_hub_download("linagora/linto-asr-ar-tn-0.1", "vosk-model.zip", local_dir=str(WORK / "vosk"))
        vdir = WORK / "vosk" / "model"
        if not vdir.exists(): zipfile.ZipFile(vz).extractall(vdir)
        mdir = next(p.parent for p in vdir.rglob("final.mdl")).parent if list(vdir.rglob("final.mdl")) else vdir
        vosk.SetLogLevel(-1); vm = vosk.Model(str(mdir))
        hyps, first_partial = [], []
        VOSK_N = min(400, len(T_AUDIO))   # bounded: Kaldi decoding on Kaggle's CPU is slow
        t0 = time.perf_counter()
        for a in T_AUDIO[:VOSK_N]:
            rec = vosk.KaldiRecognizer(vm, 16000)
            pcm = (np.clip(np.asarray(a), -1, 1) * 32767).astype(np.int16).tobytes()
            fp = None
            for i in range(0, len(pcm), 3200):            # 100 ms chunks
                rec.AcceptWaveform(pcm[i:i + 3200])
                if fp is None and json.loads(rec.PartialResult()).get("partial"): fp = (i + 3200) / 32000
            hyps.append(json.loads(rec.FinalResult()).get("text", ""))
            first_partial.append(fp)
        record("LinTO Kaldi/Vosk (native streaming)", hyps, time.perf_counter() - t0,
               {"median_first_partial_audio_s": float(np.median([f for f in first_partial if f is not None] or [np.nan]))})
    except Exception as e:
        print("Vosk baseline skipped:", repr(e)[:200])

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 61.1 MB/s eta 0:00:00:00:010:01
Note: you may need to restart the kernel to use updated packages.


vosk-model.zip:   0%|          | 0.00/542M [00:00<?, ?B/s]

LinTO Kaldi/Vosk (native streaming) n=400 WER 25.4  normWER 25.0 [22.  28.1]  CER 15.7  normCER 15.4  RTF 0.150  runaway 0


## 12. Error analysis on the test set (fine-tuned CT2 beam-3 = deployed decoder)

In [17]:
def categorize(refs, hyps, top=15):
    cats = collections.Counter(); confus = collections.Counter()
    for r, h in zip(refs, hyps):
        rn, hn = normalized_for_wer(r), normalized_for_wer(h)
        if not hn: cats["empty_output"] += 1
        if len(hn.split()) > 2 * len(rn.split()) + 3: cats["hallucination/looping (len>2x)"] += 1
        o = jiwer.process_words(rn, hn)
        rw, hw = rn.split(), hn.split()
        for ch in o.alignments[0]:
            if ch.type == "delete": cats["deleted words"] += ch.ref_end_idx - ch.ref_start_idx
            elif ch.type == "insert": cats["inserted words"] += ch.hyp_end_idx - ch.hyp_start_idx
            elif ch.type == "substitute":
                for a, b in zip(rw[ch.ref_start_idx:ch.ref_end_idx], hw[ch.hyp_start_idx:ch.hyp_end_idx]):
                    if LATIN.search(a) or LATIN.search(b):
                        cats["code-switch: " + ("script mismatch" if bool(LATIN.search(a)) != bool(LATIN.search(b)) else "Latin word wrong")] += 1
                    elif (b.startswith(a) or a.startswith(b)) and a != b:
                        cats["word segmentation (merge/split)"] += 1
                    elif 1 - jiwer.cer(a, b) >= 0.6: cats["near-miss spelling (≥60% chars right)"] += 1
                    else: cats["lexical (dialect vocab / wrong word)"] += 1
                    confus[(a, b)] += 1
    return cats, confus.most_common(top)

final_name = "fine-tuned CT2 int8_fp16 GPU beam3"
cats, confus = categorize(T_REF, hyps_by_model[final_name])
print("Error categories (word counts):")
for k, v in cats.most_common(): print(f"  {k:<42} {v}")
r = results[final_name]
print(f"normalized CER {r['norm_cer']:.1f} vs space-insensitive CER {r['nospace_cer']:.1f} → "
      f"{r['norm_cer'] - r['nospace_cer']:.1f} CER points come from word segmentation")
print("Most frequent substitutions (ref → hyp):")
for (a, b), n in confus: print(f"  {n:>3}× {a} → {b}")
rng = random.Random(SEED)
print("\nRandom examples (REF / BASE / FINE-TUNED):")
for i in rng.sample(range(len(T_REF)), min(10, len(T_REF))):
    print(f"[{T_CFG[i]}] REF : {T_REF[i]}\n        BASE: {hyps_by_model['whisper-small (base) CT2 int8_fp16 GPU beam3'][i]}\n        FT  : {hyps_by_model[final_name][i]}")

Error categories (word counts):
  near-miss spelling (≥60% chars right)      439
  lexical (dialect vocab / wrong word)       371
  deleted words                              160
  word segmentation (merge/split)            147
  inserted words                             129
  code-switch: Latin word wrong              105
  code-switch: script mismatch               78
normalized CER 11.5 vs space-insensitive CER 11.8 → -0.3 CER points come from word segmentation
Most frequent substitutions (ref → hyp):
    8× لي → الي
    6× الي → لي
    5× ساره → صاره
    4× موش → مش
    4× والا → ولا
    4× fait → faite
    3× ال → الباك
    3× اي → ايه
    3× من → ما
    3× معناتها → معنتها
    3× الكوره → نكرا
    3× نكره → الكره
    3× heures → heure
    2× لحقيقه → لقيقه
    2× هاذي → هذي

Random examples (REF / BASE / FINE-TUNED):
[TunSwitchCS] REF : كيفاش أحنا on va changer le process
        BASE: كيفيش؟ أحنا نحن نغطي العالم.
        FT  : كيفاش أحنا on va changer le process
[TunSwitchTO] R

## 13. Low-latency inference engine (`live_asr.py`) and streaming benchmark
The cell below writes the reusable engine/CLI to `/kaggle/working/live_asr.py` (download it and run locally with a microphone). The benchmark replays test audio through the engine with a **virtual real-time clock**: an audio chunk becomes available when it would have been spoken, and slow decoding accumulates as queueing delay.
* **TTFP** = first partial reaches the user − true speech onset.
* **End-of-turn latency** = final transcript reaches the user − true end of speech (includes the VAD silence wait).

In [18]:
%%writefile /kaggle/working/live_asr.py
#!/usr/bin/env python3
"""Low-latency Tunisian Derja ASR on top of faster-whisper (CTranslate2).

What this is (honestly): Whisper is an encoder-decoder model trained on <=30 s
windows; it is NOT a native streaming recognizer. This module gives a practical
"pseudo-streaming" experience:

  * Silero VAD (bundled with faster-whisper) finds speech onset and end of turn.
  * While a turn is in progress, the growing turn buffer is re-decoded every
    `step_s` seconds with greedy search -> PARTIAL hypotheses.
  * LocalAgreement-2: only the word prefix on which two consecutive partial
    decodes agree is marked "stable"; it never retracts. The rest is shown as
    "unstable" and may change.
  * When VAD sees `min_silence_ms` of silence, the turn is FINALIZED with a
    (configurable) beam search on the trimmed turn audio. Turns longer than
    `max_turn_s` are cut at the best VAD gap so the 30 s window is never exceeded.
  * Guards against silence hallucinations: no decode without VAD speech,
    minimum speech duration, faster-whisper no_speech/log-prob filters and a
    repeated-n-gram collapse.

Each partial re-decode costs one full encoder pass (Whisper pads to 30 s), so
the partial interval must be larger than the decode time on your hardware;
the engine adapts the interval automatically.

Usage (local machine with microphone):
    pip install faster-whisper "av<19" sounddevice soundfile numpy
    python live_asr.py --model ./whisper-tn-ct2                 # microphone
    python live_asr.py --model ./whisper-tn-ct2 --file talk.wav  # replay a file in real time
    python live_asr.py --model ./whisper-tn-ct2 --device cpu --compute-type int8
"""
from __future__ import annotations

import argparse
import re
import sys
import time
from dataclasses import dataclass, field
from typing import Callable, List, Optional

import numpy as np

SR = 16000


@dataclass
class StreamConfig:
    language: str = "ar"
    step_s: float = 0.6            # minimum interval between partial decodes (audio seconds)
    min_silence_ms: int = 500      # silence that ends a turn (dominates end-of-turn latency)
    vad_threshold: float = 0.5
    preroll_s: float = 0.3         # audio kept before detected onset
    tail_pad_s: float = 0.2        # audio kept after last speech when finalizing
    max_turn_s: float = 25.0       # force a cut before Whisper's 30 s window
    min_speech_s: float = 0.3      # ignore blips shorter than this
    partial_beam: int = 1
    final_beam: int = 3
    emit_partials: bool = True
    no_speech_threshold: float = 0.6
    log_prob_threshold: float = -1.0
    adaptive_step: bool = True     # widen step_s when decoding is slower than real time


def collapse_repeats(text: str, max_n: int = 4, max_rep: int = 2) -> str:
    """Collapse runs of the same 1..max_n-gram repeated more than max_rep times
    (typical Whisper looping hallucination)."""
    words = text.split()
    changed = True
    while changed:
        changed = False
        for n in range(1, max_n + 1):
            i = 0
            out: List[str] = []
            while i < len(words):
                gram = words[i:i + n]
                reps = 1
                while words[i + reps * n:i + (reps + 1) * n] == gram and len(gram) == n:
                    reps += 1
                if reps > max_rep:
                    out.extend(gram)
                    i += reps * n
                    changed = True
                else:
                    out.append(words[i])
                    i += 1
            words = out
    return " ".join(words)


def common_prefix(a: List[str], b: List[str]) -> List[str]:
    out = []
    for x, y in zip(a, b):
        if x != y:
            break
        out.append(x)
    return out


def silero_vad_fn(threshold: float = 0.5) -> Callable[[np.ndarray], List[dict]]:
    """Returns f(audio)->[{'start','end'} in samples] using faster-whisper's Silero VAD."""
    from faster_whisper.vad import VadOptions, get_speech_timestamps
    opts = VadOptions(threshold=threshold, min_speech_duration_ms=100,
                      min_silence_duration_ms=100, speech_pad_ms=0)

    def _vad(audio: np.ndarray) -> List[dict]:
        if len(audio) < SR // 10:
            return []
        return get_speech_timestamps(audio, vad_options=opts, sampling_rate=SR)
    return _vad


@dataclass
class Event:
    kind: str                  # "partial" | "final"
    text: str
    stable: str = ""
    unstable: str = ""
    audio_t: float = 0.0       # stream time (s) at which the event was produced
    turn_start: float = 0.0
    turn_end: float = 0.0
    compute_s: float = 0.0     # decode + VAD time spent producing this event


class TurnTranscriber:
    """Feed 16 kHz mono float32 chunks; receive partial/final events."""

    def __init__(self, model, cfg: Optional[StreamConfig] = None,
                 vad_fn: Optional[Callable[[np.ndarray], List[dict]]] = None):
        self.model = model
        self.cfg = cfg or StreamConfig()
        self.vad = vad_fn or silero_vad_fn(self.cfg.vad_threshold)
        self.reset()

    # ------------------------------------------------------------------ state
    def reset(self):
        self.t = 0.0                       # stream time of the end of the last fed sample
        self.ring = np.zeros(0, np.float32)
        self.turn: Optional[np.ndarray] = None
        self.turn_start = 0.0
        self.last_partial_t = 0.0
        self.prev_words: List[str] = []
        self.committed: List[str] = []
        self.step_s = self.cfg.step_s
        self.decode_times: List[float] = []

    # ---------------------------------------------------------------- decoding
    def _decode(self, audio: np.ndarray, beam: int, final: bool) -> str:
        kw = dict(language=self.cfg.language, task="transcribe", beam_size=beam,
                  condition_on_previous_text=False, vad_filter=False,
                  without_timestamps=True, no_speech_threshold=self.cfg.no_speech_threshold,
                  log_prob_threshold=self.cfg.log_prob_threshold)
        if not final:
            kw["temperature"] = 0.0          # no fallback loop for partials (latency)
        segments, _ = self.model.transcribe(audio, **kw)
        text = " ".join(s.text.strip() for s in segments).strip()
        return collapse_repeats(re.sub(r"\s+", " ", text))

    # ------------------------------------------------------------------- feed
    def feed(self, chunk: np.ndarray) -> List[Event]:
        chunk = np.asarray(chunk, dtype=np.float32).reshape(-1)
        self.t += len(chunk) / SR
        events: List[Event] = []
        t0 = time.perf_counter()
        if self.turn is None:
            keep = int((self.cfg.preroll_s + 0.7) * SR)
            self.ring = np.concatenate([self.ring, chunk])[-keep:]
            tail = self.ring[-int(0.5 * SR):]
            if self.vad(tail):
                self.turn = self.ring[-int((self.cfg.preroll_s + 0.5) * SR):].copy()
                self.turn_start = self.t - len(self.turn) / SR
                self.last_partial_t = self.t
                self.prev_words, self.committed = [], []
                self.ring = np.zeros(0, np.float32)
            return events
        self.turn = np.concatenate([self.turn, chunk])
        turn_len = len(self.turn) / SR

        # end-of-turn check on the tail window only (cheap)
        win_s = self.cfg.min_silence_ms / 1000 + 0.6
        tail = self.turn[-int(win_s * SR):]
        spans = self.vad(tail)
        silence_s = (len(tail) - (spans[-1]["end"] if spans else 0)) / SR
        if silence_s >= self.cfg.min_silence_ms / 1000 and turn_len > win_s:
            events += self._finalize(t0)
            return events
        if turn_len >= self.cfg.max_turn_s:
            events += self._force_cut(t0)
            return events

        if self.cfg.emit_partials and self.t - self.last_partial_t >= self.step_s:
            d0 = time.perf_counter()
            words = self._decode(self.turn, self.cfg.partial_beam, final=False).split()
            dt = time.perf_counter() - d0
            self.decode_times.append(dt)
            if self.cfg.adaptive_step:   # keep partial decoding below real time
                self.step_s = max(self.cfg.step_s, 1.5 * dt)
            agreed = common_prefix(self.prev_words, words)
            if len(agreed) > len(self.committed) and agreed[:len(self.committed)] == self.committed:
                self.committed = agreed
            self.prev_words = words
            stable = " ".join(self.committed)
            unstable = " ".join(words[len(self.committed):]) if words[:len(self.committed)] == self.committed else " ".join(words)
            self.last_partial_t = self.t
            events.append(Event("partial", (stable + " " + unstable).strip(), stable, unstable,
                                audio_t=self.t, turn_start=self.turn_start,
                                compute_s=time.perf_counter() - t0))
        return events

    def _emit_final(self, audio: np.ndarray, start: float, end: float, t0: float) -> List[Event]:
        spans = self.vad(audio)
        speech = sum(s["end"] - s["start"] for s in spans) / SR
        text = ""
        if speech >= self.cfg.min_speech_s:
            text = self._decode(audio, self.cfg.final_beam, final=True)
        if not text:
            return []
        return [Event("final", text, text, "", audio_t=self.t, turn_start=start, turn_end=end,
                      compute_s=time.perf_counter() - t0)]

    def _finalize(self, t0: float) -> List[Event]:
        spans = self.vad(self.turn)
        last_end = spans[-1]["end"] if spans else len(self.turn)
        cut = min(len(self.turn), last_end + int(self.cfg.tail_pad_s * SR))
        audio, rest = self.turn[:cut], self.turn[cut:]
        start, end = self.turn_start, self.turn_start + last_end / SR
        self.turn = None
        self.ring = rest[-int((self.cfg.preroll_s + 0.7) * SR):]
        self.step_s = self.cfg.step_s
        return self._emit_final(audio, start, end, t0)

    def _force_cut(self, t0: float) -> List[Event]:
        look = int(4.0 * SR)
        tail = self.turn[-look:]
        spans = self.vad(tail)
        offset = len(self.turn) - len(tail)
        cut = len(self.turn)
        gaps = [(spans[i]["end"] + spans[i + 1]["start"]) // 2 for i in range(len(spans) - 1)]
        if gaps:
            cut = offset + gaps[-1]
        audio, rest = self.turn[:cut], self.turn[cut:]
        start = self.turn_start
        events = self._emit_final(audio, start, start + cut / SR, t0)
        # continue the same speech stream as a new turn
        self.turn = rest
        self.turn_start = start + cut / SR
        self.prev_words, self.committed = [], []
        self.last_partial_t = self.t
        return events

    def flush(self) -> List[Event]:
        if self.turn is None:
            return []
        return self._finalize(time.perf_counter())


def simulate_stream(model, audio: np.ndarray, cfg: Optional[StreamConfig] = None,
                    chunk_s: float = 0.1, vad_fn=None):
    """Replay `audio` as if it arrived in real time, using a virtual clock.

    Audio chunk k becomes available at wall time = its stream end time. The
    engine processes it as soon as it is available AND the previous call has
    finished, so slow decoding accumulates as queueing delay. Every event gets
    `emit_wall`, the virtual wall-clock time it would reach the user.
    Returns (events, total_compute_s)."""
    eng = TurnTranscriber(model, cfg, vad_fn)
    step = int(chunk_s * SR)
    wall = 0.0
    out = []
    total = 0.0
    for i in range(0, len(audio), step):
        chunk = audio[i:i + step]
        avail = (i + len(chunk)) / SR
        t0 = time.perf_counter()
        evs = eng.feed(chunk)
        dt = time.perf_counter() - t0
        total += dt
        wall = max(wall, avail) + dt
        for e in evs:
            out.append((wall, e))
    t0 = time.perf_counter()
    evs = eng.flush()
    dt = time.perf_counter() - t0
    total += dt
    wall = max(wall, len(audio) / SR) + dt
    out += [(wall, e) for e in evs]
    return out, total


# ---------------------------------------------------------------------- CLI
def _load_model(path: str, device: str, compute_type: str):
    from faster_whisper import WhisperModel
    return WhisperModel(path, device=device, compute_type=compute_type)


def _print_event(e: Event):
    if e.kind == "partial":
        sys.stdout.write("\r\033[K… " + e.stable + (" \033[2m" + e.unstable + "\033[0m" if e.unstable else ""))
        sys.stdout.flush()
    else:
        sys.stdout.write("\r\033[K[%.1f–%.1fs] %s\n" % (e.turn_start, e.turn_end, e.text))
        sys.stdout.flush()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help="CTranslate2 model folder (faster-whisper format)")
    ap.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    ap.add_argument("--compute-type", default=None, help="int8 (CPU) / int8_float16 (GPU) / float16 ...")
    ap.add_argument("--file", default=None, help="WAV/FLAC file to replay in real time instead of the mic")
    ap.add_argument("--step", type=float, default=0.6, help="partial decode interval (s)")
    ap.add_argument("--min-silence-ms", type=int, default=500)
    ap.add_argument("--final-beam", type=int, default=3)
    ap.add_argument("--no-partials", action="store_true")
    a = ap.parse_args()

    device = a.device
    if device == "auto":
        try:
            import ctranslate2
            device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
        except Exception:
            device = "cpu"
    ctype = a.compute_type or ("int8_float16" if device == "cuda" else "int8")
    model = _load_model(a.model, device, ctype)
    cfg = StreamConfig(step_s=a.step, min_silence_ms=a.min_silence_ms,
                       final_beam=a.final_beam, emit_partials=not a.no_partials)
    eng = TurnTranscriber(model, cfg)
    print(f"Model loaded on {device} ({ctype}). Speak (Ctrl+C to stop)…", file=sys.stderr)

    if a.file:
        import soundfile as sf
        audio, sr = sf.read(a.file, dtype="float32", always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != SR:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=SR)
        step = int(0.1 * SR)
        t_start = time.perf_counter()
        for i in range(0, len(audio), step):
            target = t_start + (i + step) / SR
            time.sleep(max(0.0, target - time.perf_counter()))
            for e in eng.feed(audio[i:i + step]):
                _print_event(e)
        for e in eng.flush():
            _print_event(e)
        return

    import queue
    import sounddevice as sd
    q: "queue.Queue[np.ndarray]" = queue.Queue()

    def cb(indata, frames, t, status):
        q.put(indata[:, 0].copy())

    with sd.InputStream(samplerate=SR, channels=1, dtype="float32", blocksize=int(0.1 * SR), callback=cb):
        try:
            while True:
                chunk = q.get()
                while not q.empty() and len(chunk) < SR:   # catch up if decoding lagged
                    chunk = np.concatenate([chunk, q.get()])
                for e in eng.feed(chunk):
                    _print_event(e)
        except KeyboardInterrupt:
            for e in eng.flush():
                _print_event(e)


if __name__ == "__main__":
    main()

Writing /kaggle/working/live_asr.py


In [19]:
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import importlib, live_asr; importlib.reload(live_asr)
from live_asr import StreamConfig, simulate_stream

def stream_bench(fw, cfg, items, gap_s=0.8, lead_s=0.5, vad_fn=None):
    # Each item = list of utterance waves played back-to-back with silence gaps.
    ttfp, eot, part_ms, final_ms, rtf, dup = [], [], [], [], [], 0
    noise = lambda s: (0.002 * np.random.default_rng(0).standard_normal(int(s * 16000))).astype(np.float32)
    for utts in items:
        pieces, truth, t = [noise(lead_s)], [], lead_s
        for w in utts:
            pieces.append(np.asarray(w, np.float32)); truth.append((t, t + len(w) / 16000)); t += len(w) / 16000
            pieces.append(noise(gap_s)); t += gap_s
        pieces.append(noise(1.0)); audio = np.concatenate(pieces)
        evs, comp = simulate_stream(fw, audio, cfg, vad_fn=vad_fn)
        rtf.append(comp / (len(audio) / 16000))
        finals = [(w, e) for w, e in evs if e.kind == "final"]
        for (st, en) in truth:
            p = [w for w, e in evs if e.kind == "partial" and e.text and e.turn_start <= en and e.audio_t >= st]
            if p: ttfp.append(min(p) - st)
            f = [w for w, e in finals if e.turn_start <= en and e.turn_end >= st - 0.3]
            if f: eot.append(min(f) - en)
        part_ms += [1000 * e.compute_s for _, e in evs if e.kind == "partial"]
        final_ms += [1000 * e.compute_s for _, e in finals]
        words = " ".join(e.text for _, e in finals).split()
        dup += sum(1 for i in range(len(words) - 3) if words[i:i + 2] == words[i + 2:i + 4])
    pct = lambda v, q: float(np.percentile(v, q)) if v else float("nan")
    return {"n_streams": len(items), "n_turns": sum(len(u) for u in items),
            "ttfp_p50_s": pct(ttfp, 50), "ttfp_p95_s": pct(ttfp, 95),
            "end_of_turn_p50_s": pct(eot, 50), "end_of_turn_p95_s": pct(eot, 95),
            "partial_decode_p50_ms": pct(part_ms, 50), "partial_decode_p95_ms": pct(part_ms, 95),
            "final_decode_p50_ms": pct(final_ms, 50), "final_decode_p95_ms": pct(final_ms, 95),
            "rtf_mean": float(np.mean(rtf)), "repeated_bigram_events": dup}

durs = np.array(test_ds["duration"]); idx = np.arange(len(durs)); rs = np.random.default_rng(SEED)
pick = lambda lo, hi, n: list(rs.permutation(idx[(durs >= lo) & (durs < hi)])[:n])
k = 4 if SMOKE else 25
buckets = {"short <4s": [[T_AUDIO[i]] for i in pick(0, 4, k)],
           "medium 4-12s": [[T_AUDIO[i]] for i in pick(4, 12, k)],
           "long 12-30s": [[T_AUDIO[i]] for i in pick(12, 30, k)]}
multi = list(rs.permutation(idx))[:6 * (2 if SMOKE else 6)]
buckets["multi-turn ~6 utts"] = [[T_AUDIO[i] for i in multi[j:j + 6]] for j in range(0, len(multi), 6)]

stream_results = {}
for dev_name, device, ctype, partials in (("GPU int8_fp16", "cuda", "int8_float16", True),
                                          ("CPU int8 partials", "cpu", "int8", True),
                                          ("CPU int8 final-only", "cpu", "int8", False)):
    fw = WhisperModel(str(CT2_DIR), device=device, compute_type=ctype, cpu_threads=min(4, CPU_COUNT))
    cfg = StreamConfig(step_s=0.6 if device == "cuda" else 1.2, min_silence_ms=500,
                       final_beam=3 if device == "cuda" else 1, emit_partials=partials)
    for bname, items in buckets.items():
        if not items or (device == "cpu" and bname.startswith("multi")): continue
        if device == "cpu": items = items[:max(2, len(items) // 3)]
        r = stream_bench(fw, cfg, items); stream_results[f"{dev_name} | {bname}"] = r
        print(f"{dev_name:<18} {bname:<18} TTFP p50 {r['ttfp_p50_s']:.2f}s p95 {r['ttfp_p95_s']:.2f}s | "
              f"EoT p50 {r['end_of_turn_p50_s']:.2f}s p95 {r['end_of_turn_p95_s']:.2f}s | RTF {r['rtf_mean']:.3f}")
    # silence / noise hallucination check
    noise_out = [e.text for _, e in simulate_stream(fw, (0.01 * np.random.default_rng(1).standard_normal(10 * 16000)).astype(np.float32), cfg)[0]]
    stream_results[f"{dev_name} | 10s noise output"] = noise_out
    print(f"{dev_name}: output on 10 s of noise:", noise_out or "none")
    del fw; gc.collect()

GPU int8_fp16      short <4s          TTFP p50 1.39s p95 2.28s | EoT p50 0.39s p95 0.98s | RTF 0.128
GPU int8_fp16      medium 4-12s       TTFP p50 0.99s p95 1.87s | EoT p50 0.79s p95 1.01s | RTF 0.220
GPU int8_fp16      long 12-30s        TTFP p50 1.01s p95 1.89s | EoT p50 -1.32s p95 1.23s | RTF 0.303
GPU int8_fp16      multi-turn ~6 utts TTFP p50 1.24s p95 2.38s | EoT p50 0.39s p95 1.00s | RTF 0.217
GPU int8_fp16: output on 10 s of noise: none
CPU int8 partials  short <4s          TTFP p50 4.69s p95 5.48s | EoT p50 3.37s p95 5.06s | RTF 1.054
CPU int8 partials  medium 4-12s       TTFP p50 5.22s p95 6.90s | EoT p50 4.28s p95 7.81s | RTF 1.139
CPU int8 partials  long 12-30s        TTFP p50 5.18s p95 7.10s | EoT p50 1.59s p95 9.33s | RTF 1.161
CPU int8 partials: output on 10 s of noise: none
CPU int8 final-only short <4s          TTFP p50 nans p95 nans | EoT p50 3.12s p95 4.18s | RTF 0.800
CPU int8 final-only medium 4-12s       TTFP p50 nans p95 nans | EoT p50 3.90s p95 4.63s | RTF 0.52

## 14. Report and packaging

In [20]:
gpu_mem = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
report = {"mode": RUN_MODE, "hardware": {"gpus": GPU_NAMES, "nvidia_smi": gpu_mem, "cpus": CPU_COUNT},
          "base_model": BASE_MODEL, "train_sources": TRAIN_SOURCES, "eval_configs": EVAL_CONFIGS,
          "dev_utts": len(dev_ds), "test_utts": len(test_ds), "test_audio_min": audio_s / 60,
          "plan": plan, "train_hours": train_hours, "best_checkpoint": best_ckpt,
          "dev_history": [{k: h[k] for k in ("step", "eval_norm_wer", "eval_norm_cer", "eval_loss")} for h in history],
          "training_stream_stats": dict(tstats), "leakage_audit": leak_audit, "pilot_zero_shot": pilot,
          "ft_pilot": ft_pilot, "test_results": results, "streaming": stream_results,
          "ct2_model_MB": dir_mb(CT2_DIR), "decoding": "greedy unless 'beam' in name; language=ar task=transcribe; no prompt",
          "session_hours": elapsed_h()}
(WORK / "evaluation_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str))

cols = ["wer", "norm_wer", "cer", "norm_cer", "nospace_cer", "rtf"]
lines = ["| Model | n | WER | norm WER (95% CI) | CER | norm CER | no-space CER | RTF |", "|---|---|---|---|---|---|---|---|"]
for name, s in results.items():
    ci = s.get("norm_wer_ci95"); ci = f" ({ci[0]:.1f}–{ci[1]:.1f})" if ci else ""
    lines.append(f"| {name} | {s.get('n_utts')} | {s['wer']:.1f} | {s['norm_wer']:.1f}{ci} | {s['cer']:.1f} | {s['norm_cer']:.1f} | "
                 f"{s.get('nospace_cer', float('nan')):.1f} | {s['rtf']:.3f} |")
table = "\n".join(lines)
(WORK / "evaluation_table.md").write_text(table + "\n")
print(table)

for z in ("whisper-tn-v3-ct2", "whisper-tn-v3-hf"):
    (WORK / f"{z}.zip").unlink(missing_ok=True)
final_only = WORK / "whisper-tn-v3-final"; final_only.mkdir(exist_ok=True)
for f in OUT_DIR.iterdir():
    if f.is_file(): shutil.copy2(f, final_only / f.name)
shutil.make_archive(str(WORK / "whisper-tn-v3-hf"), "zip", final_only)
shutil.copy(WORK / "live_asr.py", CT2_DIR / "live_asr.py")
shutil.make_archive(str(WORK / "whisper-tn-v3-ct2"), "zip", CT2_DIR)
print("Artifacts in", WORK, ":", sorted(p.name for p in WORK.iterdir()))
print(f"Session time used: {elapsed_h():.2f} h of {SESSION_LIMIT_H} h")

| Model | n | WER | norm WER (95% CI) | CER | norm CER | no-space CER | RTF |
|---|---|---|---|---|---|---|---|
| whisper-small (base, HF fp16 greedy) | 482 | 96.3 | 94.4 (85.4–103.6) | 63.7 | 62.4 | 63.3 | 0.037 |
| fine-tuned (HF fp16 greedy) | 482 | 32.8 | 31.9 (29.9–34.2) | 13.4 | 12.9 | 13.1 | 0.025 |
| whisper-small (base) CT2 int8_fp16 GPU beam1 | 482 | 80.9 | 78.1 (75.5–80.9) | 46.5 | 44.9 | 46.1 | 0.092 |
| whisper-small (base) CT2 int8_fp16 GPU beam3 | 482 | 79.1 | 76.3 (73.8–79.0) | 44.5 | 42.9 | 44.0 | 0.078 |
| fine-tuned CT2 int8_fp16 GPU beam1 | 482 | 34.6 | 33.8 (30.5–38.6) | 15.0 | 14.6 | 14.8 | 0.042 |
| fine-tuned CT2 int8_fp16 GPU beam3 | 482 | 30.4 | 29.5 (27.5–31.6) | 12.0 | 11.5 | 11.8 | 0.044 |
| fine-tuned CT2 int8 CPU beam1 (subset) | 80 | 31.0 | 30.3 | 13.2 | 12.9 | nan | 0.889 |
| LinTO Kaldi/Vosk (native streaming) | 400 | 25.4 | 25.0 (22.0–28.1) | 15.7 | 15.4 | 15.9 | 0.150 |
Artifacts in /kaggle/working : ['.session_start', '.virtual_documents', '__pycach

## Limitations
* **Leakage:** eval sentences are block-listed from training, but there are no speaker IDs, so the same TunSwitch-TO speakers (89 in total) can appear in train and test; a speaker-disjoint, real-world recording set from your target users remains the only fully trustworthy benchmark.
* **Spelling:** Derja has no orthography; even the normalized WER penalizes legitimate spelling choices (e.g. `ضاهرلي`/`ظهر لي`). Use CER and the space-insensitive CER alongside WER.
* **Code-switching:** references write French/English words sometimes in Latin and sometimes in Arabic script; the model cannot know which convention a given annotator used.
* **Streaming:** the engine is turn-based pseudo-streaming. Partials are re-decodes of the growing turn (one 30-s encoder pass each); end-of-turn latency is bounded below by `min_silence_ms`. A truly streaming architecture (e.g. the LinTO Kaldi/Vosk model benchmarked above, or a CTC/transducer model) gives partials with lower compute but, in general, needs an external LM for competitive accuracy.
* **Hardware:** budgets are computed for the GPU found at runtime; numbers do not transfer between T4, P100 and T4×2 or to your deployment CPU — re-run the benchmark cells there (`python live_asr.py --file …` locally).